In [1]:
from GradientGang.Pipeline.DataLoader.DataLoader import DataModule

%load_ext autoreload
%autoreload 2

In [2]:
params = {
    'data_dir': "../dataset/PirateProcessed/",
    'train_file_name': "pirate_pain_train.csv",
    'train_file_name_labels': "pirate_pain_train_labels.csv",
    'test_file_name': "pirate_pain_test.csv",
    'train_global_features_file': "train_global_features.csv",
    'test_global_features_file': "test_global_features.csv",
    'batch_size': 32,
    'num_workers': 0,
    'val_split': 0.1,
    'shuffle': True,
    'use_kfold': True,
    'n_folds': 4,
    'use_windowing': False,  # Disable DataLoader windowing
}

dataLoader = DataModule(params=params)

# Setup for K-fold (loads labeled training data)
dataLoader.setup(stage='fit')

print("=" * 70)
print("TEST 1: K-Fold WITHOUT test data inclusion")
print("=" * 70)

# Setup fold 0 WITHOUT including test data
dataLoader.setup_fold(fold_idx=0, include_test_in_train=False)

trainLoader = dataLoader.train_dataloader()
valLoader = dataLoader.val_dataloader()

print(f"✓ Fold 0 setup complete (without test data)")
print(f"  Train batches: {len(trainLoader)}")
print(f"  Val batches: {len(valLoader)}")
print(f"  Train dataset size: {len(dataLoader.train_dataset)}")
print(f"  Val dataset size: {len(dataLoader.val_dataset)}")

print("\n" + "=" * 70)
print("TEST 2: K-Fold WITH test data inclusion (for Autoencoder)")
print("=" * 70)

# Setup fold 1 WITH including test data
dataLoader.setup_fold(fold_idx=1, include_test_in_train=True)

trainLoader_with_test = dataLoader.train_dataloader()
valLoader2 = dataLoader.val_dataloader()

print(f"✓ Fold 1 setup complete (with test data)")
print(f"  Train batches: {len(trainLoader_with_test)}")
print(f"  Val batches: {len(valLoader2)}")
print(f"  Train dataset size: {len(dataLoader.train_dataset)}")
print(f"  Val dataset size: {len(dataLoader.val_dataset)}")

# Check if test dataset was loaded
if hasattr(dataLoader, 'test_dataset') and dataLoader.test_dataset is not None:
    print(f"  Test dataset size: {len(dataLoader.test_dataset)}")
    print(f"\n✓ SUCCESS: Test dataset was automatically loaded!")
else:
    print(f"\n✗ FAIL: Test dataset was NOT loaded!")

print("\n" + "=" * 70)
print("SUMMARY")
print("=" * 70)
print(f"Without test data: {len(trainLoader)} batches")
print(f"With test data:    {len(trainLoader_with_test)} batches")
print(f"Difference:        {len(trainLoader_with_test) - len(trainLoader)} batches")
print("\nExpected behavior: 'With test data' should have MORE batches")
print("=" * 70)

TEST 1: K-Fold WITHOUT test data inclusion
✓ Fold 0 setup complete (without test data)
  Train batches: 16
  Val batches: 6
  Train dataset size: 495
  Val dataset size: 166

TEST 2: K-Fold WITH test data inclusion (for Autoencoder)
✓ Fold 1 setup complete (with test data)
  Train batches: 57
  Val batches: 6
  Train dataset size: 1820
  Val dataset size: 165
  Test dataset size: 1324

✓ SUCCESS: Test dataset was automatically loaded!

SUMMARY
Without test data: 16 batches
With test data:    57 batches
Difference:        41 batches

Expected behavior: 'With test data' should have MORE batches
✓ Fold 1 setup complete (with test data)
  Train batches: 57
  Val batches: 6
  Train dataset size: 1820
  Val dataset size: 165
  Test dataset size: 1324

✓ SUCCESS: Test dataset was automatically loaded!

SUMMARY
Without test data: 16 batches
With test data:    57 batches
Difference:        41 batches

Expected behavior: 'With test data' should have MORE batches


In [3]:
# Test that batches load correctly
print("Testing batch loading from training data (with test included)...")
for batch in trainLoader_with_test:
    (time_series, global_features), labels = batch
    print(f"✓ Batch loaded successfully")
    print(f"  Time Series Shape: {time_series.shape}")
    print(f"  Global Features Shape: {global_features.shape}")
    print(f"  Labels Shape: {labels.shape}")
    print(f"  Labels (first 5): {labels[:5].tolist()}")
    print(f"  Note: -1 indicates unlabeled samples (from test set)")
    break

Testing batch loading from training data (with test included)...
✓ Batch loaded successfully
  Time Series Shape: torch.Size([32, 34, 160])
  Global Features Shape: torch.Size([32, 32])
  Labels Shape: torch.Size([32])
  Labels (first 5): [-1, 0, -1, -1, -1]
  Note: -1 indicates unlabeled samples (from test set)


## Test Results

**Expected behavior:**
1. ✅ First test (without test data): Uses only labeled training samples for that fold
2. ✅ Second test (with test data): Automatically loads test dataset and concatenates it with training data
3. ✅ Training batches should increase when test data is included
4. ✅ Test dataset contains unlabeled samples (labels = -1)

**What this confirms:**
- `setup_fold()` now automatically loads the test dataset when `include_test_in_train=True`
- No need to call `setup(stage="test")` manually from FinalPipeline
- Test data is correctly concatenated with training data for autoencoder training